# Week 42, Day 4: Homeostatic plasticity and synaptic scaling

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w042/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Restoring the rate

Write run(W, x, steps): on each step compute the 50 rates y = max(0, W x), update a running average a by (y - a)/20, and multiply each neuron's row of weights by 1 + (target - a)/(target tau). Run to steady state, then cut the input to 30% (TTX) and run again.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1998)
N, M = 50, 100
W0 = r.uniform(0, 0.1, (N, M))
x = r.uniform(0, 10, M)

def run(W, x, steps, target=5.0, tau=200.0):
    a = np.zeros(N); hist = []
    for t in range(steps):
        y = np.maximum(0, W @ x)
        a += (y - a) / 20
        ...
        hist.append(y.mean())
    return W, np.array(hist)

W1, h1 = run(W0, x, 3000)
W2, h2 = run(W1, 0.3 * x, 3000)
print(f"before TTX {h1[-1]:.2f} Hz; just after {h2[0]:.2f} Hz; after scaling {h2[-1]:.2f} Hz; weights multiplied by {np.mean(W2 / W1):.2f}")
plt.plot(np.r_[h1, h2]); plt.axvline(3000, color="k", ls=":"); plt.xlabel("step"); plt.ylabel("mean rate (Hz)"); plt.show()

In [ ]:
import numpy as np
def test_restored():
    "after the input drop the rate returns to within 2% of the 5 Hz target"
    assert abs(h2[-1] - 5) < 0.1 and h2[0] < 2

def test_ratios():
    "every weight was multiplied by the same factor"
    ratio = W2 / W1
    assert np.std(ratio) / np.mean(ratio) < 1e-6 and np.mean(ratio) > 2

test_restored()
test_ratios()
print("All checks passed")

### Exercise: The rank-order test

Sort 400 control and 400 TTX amplitudes. Fit t = a c through the origin (a = c·t / c·c) and t = c + b (b = mean of t - c). Report both RMS errors.

In [ ]:
r2 = np.random.default_rng(36103)
ctrl = r2.lognormal(np.log(14), 0.35, 400)        # pA
ttx = 1.5 * r2.lognormal(np.log(14), 0.35, 400)   # an independent sample, scaled by 1.5

c, t = np.sort(ctrl), np.sort(ttx)
a_fit = ...
b_fit = ...
rms_mult = np.sqrt(np.mean((t - a_fit * c) ** 2))
rms_add = np.sqrt(np.mean((t - c - b_fit) ** 2))
print(f"multiplicative: slope {a_fit:.2f}, RMS error {rms_mult:.2f} pA;  additive: offset {b_fit:.1f} pA, RMS error {rms_add:.2f} pA")
plt.plot(c, t, ".", ms=3); plt.plot(c, a_fit * c, label="multiplicative"); plt.plot(c, c + b_fit, label="additive")
plt.xlabel("control amplitude (pA), ranked"); plt.ylabel("TTX amplitude (pA), ranked"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_slope():
    "the fitted scaling factor is close to the true 1.5"
    assert abs(a_fit - 1.5) < 0.1

def test_better():
    "the multiplicative fit leaves less than half the error of the additive one"
    assert rms_mult < 0.5 * rms_add

test_slope()
test_better()
print("All checks passed")

### Exercise: How fast must homeostasis be?

Complete hebb(tau): after the Hebbian step, update the running rate a by (y - a)/10 and, if tau is not None, multiply all weights by 1 + (target - a)/(target tau). Run it for tau None, 1000, 20 and 5.

In [ ]:
r3 = np.random.default_rng(7)
M3 = 20
xbar = np.r_[np.full(10, 2.02), np.full(10, 2.02)]   # mean input rates

def sample():
    c = r3.normal(); x = np.empty(M3)
    x[:10] = np.maximum(0, 2 + 1.0 * c + 0.5 * r3.normal(size=10))   # correlated group
    x[10:] = np.maximum(0, 2 + 1.1 * r3.normal(size=10))             # independent group
    return x

def hebb(tau, steps=20000, eta=5e-4, target=5.0):
    w = np.full(M3, 0.12); a = w @ np.full(M3, 2.0)
    for t in range(steps):
        x = sample(); y = max(0, w @ x)
        w = w + eta * (x - xbar) * y
        ...
        w = np.maximum(w, 0)
        if y > 1e4:
            return w, y, t
    return w, a, steps

res = {tau: hebb(tau) for tau in [None, 1000, 20, 5]}
for tau, (w, a, t) in res.items():
    print(f"tau {str(tau):>4}: rate {a:9.1f} Hz after {t} steps; mean weight correlated {w[:10].mean():.3f}, independent {w[10:].mean():.3f}")

In [ ]:
import numpy as np
def test_runaway():
    "Hebbian learning alone runs away"
    assert res[None][1] > 1000

def test_slow():
    "slow scaling (tau 1000) leaves the rate far above the target"
    assert res[1000][1] > 15

def test_fast():
    "fast scaling (tau 5) holds the rate near 5 Hz and keeps the correlated inputs stronger"
    w, a, _ = res[5]
    assert abs(a - 5) < 1 and w[:10].mean() > 5 * w[10:].mean()

test_runaway()
test_slow()
test_fast()
print("All checks passed")